In [1]:
# asyncpg: 비동기, httpx: 외부 API(http)비동기 호출
%pip install asyncpg httpx python-dotenv
%pip install requests


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# 테스트용 초기 state 만들기
from core.state import make_initial_state
from core.mocks import mock_user_input

initial_state = make_initial_state(mock_user_input)

In [4]:
# [노드] validate_input
# 사용자 입력 -> 좌표 변환, 식사 시간 계산 (전처리)
from nodes.validate_input import validate_input

validate_result = validate_input(initial_state)
# print("validate_input 결과:", validate_result)


In [5]:
# [노드] collect_candidate_pool
# kakao Local API로 raw 후보 풀 수집 + PostgreSQL
from nodes.collect_candidate_pool import collect_candidate_pool

candidates_result = await collect_candidate_pool(validate_result)
#print("candidates_result 결과:", candidates_result)


In [6]:
from nodes.first_filter_candidates import first_filter_candidates

filter_result = first_filter_candidates(candidates_result, debug=True)

print(f"\n📌 warnings:")
for w in filter_result["warnings"]:
    print(f"   - {w}")


📦 시작
   ──────────────────────────────────────────────────
   ✂️  제거: 0개  |  남은: 276개
   📊 카테고리: :110  FD6:69  CE7:57  AD5:13  AT4:9  PK6:7
🔍 전체 장소 목록:
01. [CE7] 오프온 | 음식점 > 카페
02. [CE7] 듀플릿 | 음식점 > 카페 > 테마카페 > 디저트카페
03. [CE7] 부산바다샌드 해운대본점 | 음식점 > 카페 > 테마카페 > 디저트카페
04. [CE7] 코오리마찌 | 음식점 > 카페 > 테마카페 > 디저트카페
05. [CE7] 모루씨 해리단길점 | 음식점 > 카페 > 커피전문점
06. [CE7] 스타벅스 해운대역점 | 음식점 > 카페 > 커피전문점 > 스타벅스
07. [CE7] 로우앤스윗 해운대카페 | 음식점 > 카페 > 커피전문점
08. [CE7] 장인더 해운대 해리단길 | 음식점 > 카페
09. [CE7] 카페플럼피 | 음식점 > 카페 > 커피전문점
10. [CE7] 블랙업커피 해운대점 | 음식점 > 카페 > 커피전문점
11. [CE7] 비커피 | 음식점 > 카페 > 커피전문점
12. [CE7] 워킹홀리데이 해운대 | 음식점 > 카페
13. [CE7] 오설록 티하우스 해운대점 | 음식점 > 카페 > 오설록 티하우스
14. [CE7] 제스터 | 음식점 > 카페
15. [CE7] 카페히토 | 음식점 > 카페
16. [CE7] 홈즈앤루팡 해운대점 | 가정,생활 > 여가시설 > 보드카페 > 홈즈앤루팡
17. [CE7] 하브커피 | 음식점 > 카페 > 커피전문점
18. [CE7] 브레이크아웃이스케이프 해운대본점 | 음식점 > 카페 > 테마카페
19. [CE7] 금송덕미 해리단길점 | 음식점 > 카페 > 테마카페 > 디저트카페
20. [CE7] 해운대옛날팥빙수단팥죽 | 음식점 > 카페 > 테마카페 > 디저트카페
21. [CE7] 프라한 해운대점 | 음식점 > 카페 > 테마카페
22. [CE7] 스타벅스 하버타운점 | 음식점 > 카

In [7]:
from nodes.second_filter_candidates import second_filter_candidates

test_state = {
    "filtered_candidates": filter_result["filtered_candidates"],
    "user_input": mock_user_input,
    "warnings": [],
}

enrich_result = await second_filter_candidates(test_state)

print(f"⚠️ warnings: {enrich_result['warnings']}")
print(f"✅ step: {enrich_result['step']}")
print(f"📦 보강된 장소 수: {len(enrich_result['filtered_candidates'])}")
print(f"📊 scored_candidates 수: {len(enrich_result['scored_candidates'])}")
print(f"🏆 shortlist 수: {len(enrich_result['shortlist'])}")

print("\n=== 전체 scored_candidates (50개) ===")

for i, s in enumerate(enrich_result["scored_candidates"], 1):
    p = s["place"]

    print(f"""
{i}위. {p['name']}
  분위기: {p.get('atmosphere')}
  추천대상: {p.get('best_for')}
  활동: {p.get('place_tags')}
  재방문의사: {p.get('revisit_intent')}
  한줄요약: {p.get('summary')}
  mood_score: {s['mood_score']}
  activity_score: {s['activity_score']}
  party_fit_score: {s['party_fit_score']}
  total_score: {s['total_score']}
""")


print("\n=== shortlist (최종 30개) ===")

for i, s in enumerate(enrich_result["shortlist"], 1):
    p = s["place"]

    print(f"""
{i}위. {p['name']}
  total_score: {s['total_score']}
  bucket: {p.get('bucket')}
""")

⏱  네이버 블로그: 11.6초 (50개)
⏱  LLM 보강: 17.9초 (50개)
⚠️ warnings: []
✅ step: enriched
📦 보강된 장소 수: 50
📊 scored_candidates 수: 50
🏆 shortlist 수: 30

=== 전체 scored_candidates (50개) ===

1위. 강릉송정해변막국수 부산분점
  분위기: ['활기찬', '이색', '깔끔한']
  추천대상: ['연인', '친구']
  활동: ['막국수', '부산', '해변']
  재방문의사: high
  한줄요약: 시원한 막국수와 쫄깃한 수육이 일품.
  mood_score: 20
  activity_score: 0
  party_fit_score: 40
  total_score: 90


2위. 나가하마만게츠
  분위기: ['이색', '힐링', '깔끔한']
  추천대상: ['연인', '가족']
  활동: ['일식', '라멘', '미슐랭']
  재방문의사: high
  한줄요약: 미슐랭 맛집으로 특별한 라멘 경험.
  mood_score: 20
  activity_score: 0
  party_fit_score: 40
  total_score: 90


3위. 워킹홀리데이 해운대
  분위기: ['힐링', '로맨틱']
  추천대상: ['연인']
  활동: ['오션뷰', '카페']
  재방문의사: high
  한줄요약: 바다뷰가 멋진 힐링 카페입니다.
  mood_score: 10
  activity_score: 10
  party_fit_score: 40
  total_score: 90


4위. 씨라이프 부산아쿠아리움
  분위기: ['힐링', '감성']
  추천대상: ['가족', '연인']
  활동: ['아쿠아리움', '바다', '관람']
  재방문의사: high
  한줄요약: 해양 생물을 관람할 수 있는 아쿠아리움.
  mood_score: 10
  activity_score: 10
  party_fit_score: 40
  total_score: 90



In [8]:
# # LangGraph 그래프 빌드
# from langgraph.graph import StateGraph, START, END
#
# graph_builder = StateGraph(TravelState)
#
# # 노드 등록
# graph_builder.add_node("validate_input", validate_input)
# graph_builder.add_node("fetch_candidates", fetch_candidates)
# graph_builder.add_node("filter_candidates", filter_candidates)
# graph_builder.add_node("enrich_candidates", enrich_candidates)
#
# # 엣지 (직선 연결)
# graph_builder.add_edge(START, "validate_input")
# graph_builder.add_edge("validate_input", "fetch_candidates")
# graph_builder.add_edge("fetch_candidates", "filter_candidates")
# graph_builder.add_edge("filter_candidates", "enrich_candidates")
# graph_builder.add_edge("enrich_candidates", END)
#
# graph = graph_builder.compile()

In [9]:
# # 그래프 실행
# state_v1 = await graph.ainvoke(initial_state)
#
# print(f"📍 위치: {state_v1['user_input']['location']}")
# print(f"📌 좌표: ({state_v1['user_input']['center_lat']}, {state_v1['user_input']['center_lng']})")
# print(f"🔍 Kakao raw 후보: {len(state_v1['candidates'])}개")
# print(f"⚠️  warnings: {state_v1['warnings']}")
# print(f"❌ errors: {state_v1['errors']}")
# print(f"✅ step: {state_v1['step']}\n")
#
# for p in filter_result["filtered_candidates"][:5]:
#     print(
#         f"  [{p.get('category_group_code', '기타')}] {p['name']} - "
#         f"{p.get('road_address_name') or p.get('address_name', '')}"
#     )

In [10]:
# from IPython.display import Image, display
# display(Image(graph.get_graph().draw_mermaid_png()))